# 📖 Notebook 1: Message Delivery & Storage

Welcome to the first notebook in our **WhatsApp System Design Lab**! 🎉

In this notebook, we'll explore how a messaging app like WhatsApp actually delivers messages
from one person to another — even when the recipient is offline.

---

## 🎯 Learning Objectives

By the end of this notebook, you will understand:

1. **How messages are stored** in a relational database
2. **How WebSockets** enable real-time message delivery
3. **The Inbox Pattern** — how offline users receive messages later
4. **Redis Pub/Sub** — how real-time notifications work
5. **Sequence numbers** — how clients detect missed messages

## 🗺️ Where This Fits

```
📓 Notebook 1: Message Delivery & Storage   <-- YOU ARE HERE
📓 Notebook 2: Read Receipts & Presence
📓 Notebook 3: Group Messaging
📓 Notebook 4: End-to-End Encryption Basics
```

## 🏗️ Architecture Overview

```
┌──────────┐     WebSocket      ┌──────────────┐
│  Alice   │◄──────────────────►│              │
│  Phone   │                    │  Chat Server │
└──────────┘                    │  (Python)    │
                                │              │
┌──────────┐     WebSocket      │              │     ┌────────────┐
│  Bob     │◄──────────────────►│              │────►│ PostgreSQL │
│  Phone   │                    │              │     │ (Storage)  │
└──────────┘                    │              │     └────────────┘
                                │              │
┌──────────┐     WebSocket      │              │     ┌────────────┐
│  Diana   │◄──────────────────►│              │────►│   Redis    │
│  Phone   │                    │              │     │ (Pub/Sub)  │
└──────────┘                    └──────────────┘     └────────────┘
```

## ⚙️ Setup

Before running this notebook, make sure the infrastructure is running:

### 1. Start Docker services

```bash
cd 06-system-designs/whatsapp
docker compose up -d
```

This starts PostgreSQL, Redis, the Chat Server, Adminer, and RedisInsight.

### 2. Select the right kernel

This notebook uses a **virtual environment** (`.venv`) with the required dependencies.

1. In VS Code, look at the **top-right** of this notebook for the kernel selector
2. Click it and choose the `.venv` kernel (from `06-system-designs/whatsapp/.venv`)
3. If you don't see it, run:
   ```bash
   cd 06-system-designs/whatsapp
   uv venv
   source .venv/bin/activate
   uv sync
   ```
4. Then reload VS Code (`Cmd+Shift+P` → "Reload Window")

### 3. Visualization tools (optional but recommended)

- **Adminer** (PostgreSQL GUI): http://localhost:8080
  - System: PostgreSQL, Server: postgres, User: demo, Password: demo, Database: whatsapp_demo
- **RedisInsight** (Redis GUI): http://localhost:5540
  - Add connection: host=localhost, port=6379

In [ ]:
# === 🔌 Connection Setup ===

import psycopg2
import psycopg2.extras
import redis
import json
import time
import threading
from websockets.sync.client import connect as ws_connect

# --- Configuration ---
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "whatsapp_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

WS_URL = "ws://localhost:8765"

# --- Helper Functions ---
def get_db():
    """Create a new database connection."""
    return psycopg2.connect(**DB_CONFIG)

def query(sql, params=None):
    """Run a SELECT query and return rows as dictionaries."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute(sql, params)
    rows = [dict(row) for row in cur.fetchall()]
    cur.close()
    conn.close()
    return rows

def execute(sql, params=None):
    """Run an INSERT/UPDATE/DELETE query."""
    conn = get_db()
    cur = conn.cursor()
    cur.execute(sql, params)
    conn.commit()
    cur.close()
    conn.close()

def get_redis():
    """Create a new Redis connection."""
    return redis.Redis(**REDIS_CONFIG)

def print_table(rows, title=""):
    """Pretty-print a list of dicts as a table."""
    if not rows:
        print("  (no rows)")
        return
    if title:
        print(f"\n📋 {title}")
        print("─" * 60)
    keys = list(rows[0].keys())
    widths = {k: max(len(str(k)), max(len(str(r.get(k, ''))) for r in rows)) for k in keys}
    header = " | ".join(str(k).ljust(widths[k]) for k in keys)
    print(f"  {header}")
    print(f"  {'─' * len(header)}")
    for row in rows:
        line = " | ".join(str(row.get(k, '')).ljust(widths[k]) for k in keys)
        print(f"  {line}")

# --- Test Connections ---
print("🔌 Testing connections...\n")

try:
    rows = query("SELECT COUNT(*) as count FROM users")
    print(f"✅ PostgreSQL: Connected! ({rows[0]['count']} users found)")
except Exception as e:
    print(f"❌ PostgreSQL: {e}")

try:
    r = get_redis()
    r.ping()
    print(f"✅ Redis: Connected!")
except Exception as e:
    print(f"❌ Redis: {e}")

try:
    ws = ws_connect(WS_URL)
    ws.send(json.dumps({"type": "connect", "user_id": 1}))
    resp = json.loads(ws.recv())
    print(f"✅ WebSocket: Connected! (server says: {resp['type']})")
    ws.close()
except Exception as e:
    print(f"❌ WebSocket: {e}")

print("\n🎉 All connections ready! Let's explore message delivery.")

---

# 🤔 The Problem: Why Is Message Delivery Hard?

Sending a text message sounds simple, right? You type "Hey!" and your friend sees it.

But think about all the things that can go wrong:

| Scenario | What happens? |
|----------|---------------|
| 📱 Bob is online | Message should appear **instantly** |
| 😴 Bob is offline | Message must be **saved** and delivered when Bob comes back |
| 📵 Bob's network drops mid-delivery | Message might be **lost** — we need to detect this |
| 💻📱 Bob has phone AND laptop | Message must reach **all** devices |
| 🔄 Messages arrive out of order | We need to **reorder** them correctly |

### 📮 Think of it like a post office

Imagine you're sending a letter to a friend:

```
📝 You write a letter (compose message)
    │
    ▼
📮 You drop it in the mailbox (send to server)
    │
    ▼
🏤 Post office receives it (server stores message)
    │
    ├──► 🏠 Friend is home? ──► Deliver immediately! ✅
    │
    └──► 🚫 Friend is away? ──► Hold in PO Box (inbox)
                                    │
                                    ▼
                               🏠 Friend comes home
                                    │
                                    ▼
                               📬 Picks up mail (sync)
                                    │
                                    ▼
                               ✅ Signs receipt (ACK)
```

This is exactly how our system works! Let's see how it's built.

---

# 🚫 Bad → ✅ Best: The Evolution of Message Delivery

Before we look at the _good_ design, let's understand **why** we chose it by walking through three designs — from naive to production-ready.

### ❌ v1 (BAD): HTTP Polling — "Did I get any new messages?"

```
Bob's phone every 5 seconds:
  GET /messages?since=123   -> Server: "nothing new"
  GET /messages?since=123   -> Server: "nothing new"
  GET /messages?since=123   -> Server: "1 new message!"
```

Problems:
- 📉 **Slow** — average 2.5s latency, up to 5s.
- 🔋 **Battery killer** — phone wakes up constantly.
- 💸 **Expensive** — 99% of requests return nothing.
- 📦 For 1B users polling every 5s → **200M requests/sec** just to ask "anything new?".

### ⚠️ v2 (BETTER): WebSocket only — no durable inbox

Replace polling with a live WebSocket. Now delivery is instant — but what if Bob is offline, or the notification drops mid-flight?

```
Alice -> Server -> (Bob offline) -> 💨 message vanishes
```

Redis pub/sub is **fire-and-forget**. No listener = no delivery. Messages get lost.

### ✅ v3 (BEST): WebSocket + Inbox table + Pub/Sub

Two layers working together:

| Layer | Technology | Role |
|-------|-----------|------|
| **Reliability** | `inbox` table in PostgreSQL | Remembers who still needs to receive each message |
| **Speed** | Redis pub/sub on `user:{id}` channels | Delivers instantly to online users |

If pub/sub drops it, the inbox still has it — the client resyncs when it reconnects. This is the design we're going to explore for the rest of the notebook. 👇


In [ ]:
# 🔢 Back-of-envelope: polling vs WebSocket at scale

users = 1_000_000_000          # 1 billion users
poll_interval_sec = 5          # check every 5 seconds
msgs_per_user_per_day = 40     # realistic WhatsApp-like average

# --- v1: HTTP polling ---
polls_per_sec = users / poll_interval_sec
real_msgs_per_sec = users * msgs_per_user_per_day / 86400
wasted_pct = (1 - real_msgs_per_sec / polls_per_sec) * 100

print('v1 (BAD) - HTTP polling')
print(f'   Requests/sec:         {polls_per_sec:>20,.0f}')
print(f'   Wasted (no new msgs): {wasted_pct:>19.2f} %')
print(f'   Avg latency:          {poll_interval_sec/2:>20.1f} s')

# --- v3: WebSocket + inbox (only work on real events) ---
print('\nv3 (BEST) - WebSocket + inbox + pub/sub')
print(f'   Writes/sec (real):    {real_msgs_per_sec:>20,.0f}')
print(f'   Wasted work:          {0:>19} %')
print(f'   Avg latency:          {"~50ms":>20}')

print(f'\n💡 Polling does ~{polls_per_sec/real_msgs_per_sec:,.0f}x more work per delivered message.')


---

# 🗄️ Exploring the Data Model

Before we send any messages, let's look at how data is organized in our database.

Our database has **6 tables**:

```
┌──────────────┐     ┌──────────────────┐     ┌──────────────┐
│    users     │     │ chat_participants │     │    chats     │
├──────────────┤     ├──────────────────┤     ├──────────────┤
│ id           │◄────│ user_id          │     │ id           │
│ username     │     │ chat_id          │───►│ name         │
│ display_name │     │ role             │     │ is_group     │
└──────────────┘     └──────────────────┘     └──────────────┘
                                                     │
┌──────────────┐     ┌──────────────────┐           │
│    inbox     │     │    messages       │           │
├──────────────┤     ├──────────────────┤           │
│ user_id      │     │ id               │           │
│ message_id   │───►│ chat_id          │◄──────────┘
│ status       │     │ sender_id        │
│ delivered_at │     │ content          │     ┌────────────────┐
└──────────────┘     │ sequence_number  │     │ chat_sequences │
                     └──────────────────┘     ├────────────────┤
                                              │ chat_id        │
                                              │ last_sequence  │
                                              └────────────────┘
```

Let's explore each one!

In [ ]:
# 👥 Let's see who's in our system
users = query("SELECT id, username, display_name FROM users ORDER BY id")
print_table(users, "Users in our system")
print("\n💡 We have 5 test users. Think of them as 5 friends with the app installed.")

In [ ]:
# 💬 Let's see the conversations (chats)
chats = query("""
    SELECT c.id, c.name, c.is_group,
           string_agg(u.username, ', ' ORDER BY u.id) as participants
    FROM chats c
    JOIN chat_participants cp ON c.id = cp.chat_id
    JOIN users u ON cp.user_id = u.id
    GROUP BY c.id, c.name, c.is_group
    ORDER BY c.id
""")
print_table(chats, "Chats (conversations)")

print("\n💡 Notice:")
print("   • Chats 1 & 2 are 1:1 chats (is_group = False, name is empty)")
print("   • Chat 3 is a group chat called 'Study Group'")

In [ ]:
# 📨 Let's see the messages that already exist
messages = query("""
    SELECT m.id, m.chat_id, u.username as sender, m.content,
           m.sequence_number as seq
    FROM messages m
    JOIN users u ON m.sender_id = u.id
    ORDER BY m.chat_id, m.sequence_number
""")
print_table(messages, "All messages in the system")

print("\n💡 Key observations:")
print("   • Each message has a sequence number (seq) WITHIN its chat")
print("   • Chat 1: Alice and Bob chatting (3 messages)")
print("   • Chat 2: Alice and Charlie chatting (2 messages)")
print("   • Chat 3: Group chat with 3 messages")

---

# 📤 Sending a Message via WebSocket

Now let's actually **send a message** through our chat system!

### What's a WebSocket?

Think of HTTP (normal web requests) like sending letters back and forth — each time you want
to say something, you write a new letter and wait for a reply.

A **WebSocket** is like a **phone call** — once connected, both sides can talk freely at any time.
This is perfect for chat apps because messages need to arrive **instantly**.

```
HTTP (like letters):              WebSocket (like a phone call):

Client ──request──► Server        Client ◄──────────► Server
Client ◄─response─ Server          (open connection, both
Client ──request──► Server           sides can send anytime)
Client ◄─response─ Server
```

### The Message Flow

Here's what happens when Alice sends a message:

```
 Alice's Phone                    Server                         Database
      │                              │                              │
      │  1. {send_message}           │                              │
      │─────────────────────────────►│                              │
      │                              │  2. Store message            │
      │                              │─────────────────────────────►│
      │                              │  3. Create inbox entries     │
      │                              │─────────────────────────────►│
      │  4. {ack, message_id}        │                              │
      │◄─────────────────────────────│                              │
      │                              │                              │
```

Let's try it! 👇

In [ ]:
# 📤 Send a message as Alice to Chat 1 (Alice <-> Bob)

# Step 1: Connect as Alice (user_id = 1)
print("📱 Connecting as Alice...")
ws = ws_connect(WS_URL)
ws.send(json.dumps({"type": "connect", "user_id": 1}))
connect_resp = json.loads(ws.recv())
print(f"   ✅ Connected! Server says: {connect_resp}")

# Step 2: Send a message to chat 1
message_content = "Hey Bob! This message was sent from the Jupyter notebook! 🚀"
print(f"\n📤 Sending message: '{message_content}'")

ws.send(json.dumps({
    "type": "send_message",
    "chat_id": 1,
    "content": message_content
}))

# Step 3: Receive the ACK (acknowledgment)
ack_resp = json.loads(ws.recv())
print(f"\n📨 Server ACK received:")
print(f"   type: {ack_resp['type']}")
print(f"   message_id: {ack_resp.get('message_id')}")
print(f"   status: {ack_resp.get('status')}")

# Save the message_id so we can clean it up later
test_message_id = ack_resp.get('message_id')

ws.close()
print("\n🔌 Disconnected.")
print("\n💡 The ACK tells Alice her message was safely stored on the server.")
print("   Without this ACK, Alice's phone would keep retrying!")

In [ ]:
# 🔍 Let's verify the message was stored in the database

if test_message_id:
    new_msg = query("""
        SELECT m.id, m.chat_id, u.username as sender, m.content,
               m.sequence_number as seq, m.server_timestamp
        FROM messages m
        JOIN users u ON m.sender_id = u.id
        WHERE m.id = %s
    """, (test_message_id,))
    print_table(new_msg, "Our new message in the database")

    print("\n✅ The message is safely stored!")
    print("   Even if the server crashes right now, this message is safe in PostgreSQL.")
    print(f"   Notice the sequence number — it's the next one in Chat 1.")
else:
    print("⚠️ No message was sent — check the previous cell.")

---

# 📬 The Inbox Pattern: Reliable Offline Delivery

Here's a crucial question: **what if Bob wasn't online when Alice sent her message?**

The server can't just forget about it! We need to **remember** that Bob hasn't received it yet.

This is where the **inbox table** comes in.

### How it works

Every time a message is sent, the server creates **one inbox entry per recipient**:

```
Alice sends "Hey!" to Chat 1 (Alice <-> Bob)

┌─────────────────────────────────────────────┐
│  messages table                              │
│  ┌────┬─────────┬──────┬──────┐             │
│  │ id │ chat_id │ from │ text │             │
│  │ 99 │    1    │ Alice│ Hey! │             │
│  └────┴─────────┴──────┴──────┘             │
└─────────────────────────────────────────────┘
                    │
                    ▼
┌─────────────────────────────────────────────┐
│  inbox table                                 │
│  ┌─────────┬────────────┬─────────┐         │
│  │ user_id │ message_id │ status  │         │
│  │  Bob(2) │     99     │ pending │  <-- Bob hasn't received it yet!
│  └─────────┴────────────┴─────────┘         │
└─────────────────────────────────────────────┘
```

When Bob comes online and receives the message, he sends an **ACK**.
The inbox entry is then updated to `delivered`.

### Why not just check the messages table?

Good question! The inbox table is important because:

1. **Speed**: Querying "give me all pending messages for Bob" is fast with an indexed inbox
2. **Per-user tracking**: In group chats, each person receives at their own pace
3. **Status tracking**: We know exactly what's `pending`, `delivered`, or `read`

Let's look at the inbox! 👇

In [ ]:
# 📬 Let's look at the inbox — who has pending messages?

inbox = query("""
    SELECT i.id, u.username as recipient, m.content as message,
           sender.username as from_user, i.status, i.created_at
    FROM inbox i
    JOIN users u ON i.user_id = u.id
    JOIN messages m ON i.message_id = m.id
    JOIN users sender ON m.sender_id = sender.id
    WHERE i.status = 'pending'
    ORDER BY i.user_id, i.created_at
""")
print_table(inbox, "Pending inbox entries (undelivered messages)")

print("\n💡 Observations:")
print("   • Bob has pending message(s) from Alice")
print("   • Diana has 3 pending messages (she hasn't opened the group chat yet!)")
print("   • The inbox tells the server EXACTLY who needs WHICH messages")

# Also check: our new message should have created an inbox entry for Bob
if test_message_id:
    new_inbox = query("""
        SELECT u.username as recipient, i.status
        FROM inbox i
        JOIN users u ON i.user_id = u.id
        WHERE i.message_id = %s
    """, (test_message_id,))
    print(f"\n📌 Our new message (id={test_message_id}) created inbox entries for:")
    for row in new_inbox:
        print(f"   • {row['recipient']} — status: {row['status']}")

---

# 📡 Redis Pub/Sub: Real-Time Delivery

The inbox handles **reliability** (making sure messages aren't lost).
But what about **speed**? When Bob IS online, we want **instant** delivery!

That's where **Redis Pub/Sub** comes in.

### What is Pub/Sub?

Imagine a **radio station**:
- The radio station **publishes** (broadcasts) a signal
- Anyone tuned in **subscribes** (listens) to that channel
- If you're not listening, you **miss** the broadcast (it's not saved)

```
         Redis Pub/Sub

  Publisher                    Subscribers
  (Server)                    (Connected users)

     📡 ─── channel: user:2 ──────► 🔊 Bob (online)     ✅ Gets it!
     │
     └── channel: user:4 ──────────► 🔇 Diana (offline)  ❌ Misses it!
                                         (but inbox has it!)
```

### Two delivery layers working together

```
┌─────────────────────────────────────────────────┐
│  Layer 1: Redis Pub/Sub (FAST but unreliable)   │
│  • Instant delivery to online users             │
│  • "At most once" — if you miss it, it's gone   │
│  • Like a live radio broadcast                  │
├─────────────────────────────────────────────────┤
│  Layer 2: Inbox (RELIABLE but slower)           │
│  • Persisted in PostgreSQL                      │
│  • Delivered on next sync                       │
│  • Like a PO Box that holds your mail           │
└─────────────────────────────────────────────────┘
```

Let's see Redis Pub/Sub in action! 👇

In [ ]:
# 📡 Redis Pub/Sub in action!
#
# We'll demonstrate how the server uses Redis to notify online users.
# Channel format: "user:{user_id}"

r = get_redis()

# --- Step 1: Set up a subscriber for Bob (user_id=2) ---
# In the real system, the server does this when a user connects.
# Here we'll do it manually to see how it works.

pubsub = r.pubsub()
pubsub.subscribe("user:2")  # Listen for messages to Bob
print("📡 Subscribed to channel 'user:2' (Bob's notification channel)")

# Consume the subscription confirmation.
#
# This line is load-bearing, not bookkeeping. `subscribe()` only WRITES the
# SUBSCRIBE command to a socket. Until Redis has actually processed it, a
# PUBLISH arriving on a different connection goes to a channel with no
# listeners and is dropped -- pub/sub keeps no backlog. Reading the
# confirmation proves the round-trip finished.
confirmation = pubsub.get_message(timeout=2)
assert confirmation and confirmation["type"] == "subscribe", (
    f"expected a subscribe confirmation before publishing, got {confirmation}"
)

# Belt and braces: ask Redis itself how many subscribers the channel has.
# (The chat server does the equivalent on connect -- it will not tell a client
#  "connected" until its own subscription is confirmed, for exactly this reason.)
subscriber_count = r.pubsub_numsub("user:2")[0][1]
print(f"   Redis reports {subscriber_count} subscriber(s) on 'user:2'")
assert subscriber_count >= 1, "nobody is listening yet -- a publish now would vanish"

# --- Step 2: Publish a test notification ---
# This simulates what the server does when someone sends Bob a message
test_notification = json.dumps({
    "type": "new_message",
    "message_id": 999,
    "chat_id": 1,
    "sender_id": 1,
    "content": "This is a test notification via Redis!",
    "sequence_number": 99
})

num_receivers = r.publish("user:2", test_notification)
assert num_receivers >= 1, (
    "PUBLISH reported zero receivers -- the notification was dropped on the floor"
)
print(f"\n📤 Published a notification to 'user:2'")
print(f"   Number of subscribers who received it: {num_receivers}")

# --- Step 3: Read the notification as Bob ---
msg = pubsub.get_message(timeout=2)
if msg and msg['type'] == 'message':
    data = json.loads(msg['data'])
    print(f"\n🔔 Bob received a notification!")
    print(f"   Type: {data['type']}")
    print(f"   Content: {data['content']}")
    print(f"   From user_id: {data['sender_id']}")
else:
    print(f"\n⚠️ No message received (msg={msg})")

pubsub.unsubscribe("user:2")
pubsub.close()

print("\n💡 Key insight: Redis Pub/Sub is FIRE-AND-FORGET.")
print("   If Bob wasn't subscribed, the notification would be lost forever.")
print("   That's why we ALSO have the inbox table as a safety net!")

---

# 🔄 Offline → Online Sync

Now let's see the full picture. Remember Diana? She's a member of the Study Group
(Chat 3) but hasn't been online. She has **3 pending messages** in her inbox.

Let's simulate Diana opening the app:

```
Diana's Phone                   Server                     Database
     │                              │                          │
     │  1. {connect, user_id: 4}    │                          │
     │────────────────────────────►│                          │
     │                              │                          │
     │  2. {connected}              │                          │
     │◄────────────────────────────│                          │
     │                              │                          │
     │  3. {sync}                   │  4. Query inbox          │
     │────────────────────────────►│────────────────────────►│
     │                              │                          │
     │  5. {new_message} x3         │◄────────────────────────│
     │◄────────────────────────────│                          │
     │                              │                          │
     │  6. {sync_complete, count:3} │                          │
     │◄────────────────────────────│                          │
     │                              │                          │
     │  7. {ack, message_id} x3     │  8. Update inbox         │
     │────────────────────────────►│────────────────────────►│
     │                              │     status -> delivered   │
```

Let's do this step by step! 👇

In [ ]:
# 🔄 Simulate Diana coming online and syncing her messages

# First, let's see what's pending for Diana
diana_pending = query("""
    SELECT i.message_id, m.content, sender.username as from_user
    FROM inbox i
    JOIN messages m ON i.message_id = m.id
    JOIN users sender ON m.sender_id = sender.id
    WHERE i.user_id = 4 AND i.status = 'pending'
    ORDER BY m.server_timestamp
""")
print(f"📬 Diana has {len(diana_pending)} pending messages before sync:")
for msg in diana_pending:
    print(f"   • [{msg['from_user']}]: {msg['content']}")

# --- Step 1: Connect as Diana ---
print("\n📱 Diana opens the app...")
ws = ws_connect(WS_URL)
ws.send(json.dumps({"type": "connect", "user_id": 4}))
connect_resp = json.loads(ws.recv())
print(f"   ✅ Connected! ({connect_resp})")

# --- Step 2: Request sync ---
print("\n🔄 Diana requests sync (give me everything I missed)...")
ws.send(json.dumps({"type": "sync"}))

# --- Step 3: Receive all pending messages ---
synced_messages = []
while True:
    resp = json.loads(ws.recv())
    if resp["type"] == "sync_complete":
        print(f"\n✅ Sync complete! Received {resp['count']} messages.")
        break
    elif resp["type"] == "new_message":
        synced_messages.append(resp)
        print(f"   📨 Message {resp['message_id']}: \"{resp['content']}\"")

# The server must hand back a backlog in (chat_id, sequence_number) order --
# NOT in timestamp order. Ordering by timestamp lets two concurrent senders in
# the same chat come back inverted; the sequence number is the only authority.
sync_keys = [(m["chat_id"], m["sequence_number"]) for m in synced_messages]
assert sync_keys == sorted(sync_keys), (
    f"sync delivered the backlog out of per-chat order: {sync_keys}"
)
print(f"   ✅ Backlog arrived in per-chat sequence order: {sync_keys}")

# --- Step 4: ACK each message ---
print("\n📝 Diana ACKs each message (confirming delivery)...")
for msg in synced_messages:
    ws.send(json.dumps({"type": "ack", "message_id": msg["message_id"]}))
    ack_resp = json.loads(ws.recv())
    print(f"   ✅ ACK'd message {msg['message_id']} -> {ack_resp['type']}")

ws.close()
print("\n🔌 Diana disconnects.")

In [ ]:
# 🔍 Let's verify Diana's inbox was updated

diana_inbox = query("""
    SELECT i.message_id, m.content, i.status, i.delivered_at
    FROM inbox i
    JOIN messages m ON i.message_id = m.id
    WHERE i.user_id = 4
    ORDER BY i.message_id
""")
print_table(diana_inbox, "Diana's inbox after sync")

print("\n💡 All messages are now 'delivered'!")
print("   The inbox entries were updated when Diana sent her ACKs.")
print("   If Diana syncs again, she won't get these messages — they're already delivered.")

---

# 🔢 Sequence Numbers: Ordering *and* Detecting Missed Messages

Imagine you're watching a TV series, and you see episodes 1, 2, 3, 5.
You'd immediately know: **"Wait, I missed episode 4!"**

That's exactly how **sequence numbers** work in our chat system — and they solve
**two different problems** that are easy to confuse:

| Problem | Symptom | Fix |
|---------|---------|-----|
| **Reordering** | You have every message, but they're in the wrong order | Sort by `sequence_number` |
| **Loss** | A message never arrived at all | Detect a gap in the sequence, re-request it |

### How it works

Each chat has a **counter** that increases by 1 for every message:

```
Chat 1 (Alice <-> Bob):

  seq=1: "Hey Bob!"           ✅ Client has this
  seq=2: "Hi Alice!"          ✅ Client has this
  seq=3: "Want coffee?"       ❌ Client missed this!
  seq=4: "Sure, when?"        ✅ Client has this
                                     │
                                     ▼
                              🚨 GAP DETECTED!
                              Client knows seq 3 is
                              missing and requests it.
```

### Why is this needed?

Even with the inbox, things can go wrong:
- Network hiccup during delivery
- A pub/sub push and a sync reply race each other to the client
- Server crashes between storing a message and notifying

### ⚠️ Per-*conversation*, not global

The counter lives in `chat_sequences`, one row per chat. There is **no global
message order** in this system, and there doesn't need to be one — nobody can
observe the relative order of two messages in two different conversations.
A global counter would mean serialising every send in the world through one
lock. Per-chat ordering is the guarantee that is both *cheap* and *sufficient*.

> 🔒 The server takes a row lock on `chat_sequences` for the whole send
> transaction (see `store_message` in `server/chat_server.py`). So within one
> chat, sequence order == commit order — they can never disagree. Two *different*
> chats still write concurrently; they lock different rows.

Let's break the ordering on purpose, then fix it. 👇

In [ ]:
# 🔢 Let's examine chat sequence numbers

sequences = query("""
    SELECT cs.chat_id,
           COALESCE(c.name, 'Chat ' || cs.chat_id::text) as chat_name,
           cs.last_sequence
    FROM chat_sequences cs
    JOIN chats c ON cs.chat_id = c.id
    ORDER BY cs.chat_id
""")
print_table(sequences, "Chat sequence counters")

print("\n💡 Each chat tracks how many messages have been sent.")
print("   These counters only go UP — they never reset or go backwards.")

# The counter must never fall behind the messages actually stored, or the next
# send would hand out a sequence number that is already taken.
for s in sequences:
    highest = query(
        "SELECT COALESCE(MAX(sequence_number), 0) AS m FROM messages WHERE chat_id = %s",
        (s['chat_id'],)
    )[0]['m']
    assert s['last_sequence'] >= highest, (
        f"chat {s['chat_id']}: counter is {s['last_sequence']} but a message "
        f"already claims sequence {highest} — the counter went backwards"
    )
print("   ✅ Every counter is >= the highest sequence actually stored.")

In [ ]:
# 🔍 Client-side ordering & gap detection
#
# The server hands out sequence numbers under a per-chat lock, so the SERVER's
# copy of a chat is always a dense 1..N run — querying it can never show a gap.
# The CLIENT is the side that gets it wrong: a pub/sub push races a sync reply,
# packets get reordered, a notification is dropped. So both checks below run
# against what the client actually RECEIVED, not against the messages table.

import random

rng = random.Random(42)   # seeded — this demo must reproduce identically

truth = query("""
    SELECT sequence_number AS seq, content
    FROM messages WHERE chat_id = 1
    ORDER BY sequence_number
""")
server_order = [m['seq'] for m in truth]
assert len(server_order) >= 3, f"need >=3 messages in chat 1 to demo this, got {server_order}"
print(f"📡 Server's view of chat 1 (authoritative): {server_order}")

# ── 1. Reproduce the failure: the messages arrive out of order ──
arrival = list(truth)
for _ in range(20):                      # keep shuffling until it really differs
    rng.shuffle(arrival)
    if [m['seq'] for m in arrival] != server_order:
        break
arrived_seqs = [m['seq'] for m in arrival]
print(f"\n📥 Order they actually hit the client's socket: {arrived_seqs}")
assert arrived_seqs != server_order, (
    f"this cell must DEMONSTRATE reordering before fixing it, got {arrived_seqs}"
)
print("   🚨 Rendering in arrival order would show a scrambled conversation:")
for m in arrival:
    print(f"      seq={m['seq']}  {m['content'][:45]}")

# ── 2. The fix: order by sequence_number, never by arrival time ──
repaired = sorted(arrival, key=lambda m: m['seq'])
repaired_seqs = [m['seq'] for m in repaired]
print(f"\n🔧 After sorting by sequence_number:       {repaired_seqs}")
assert repaired_seqs == server_order, (
    f"sorting by sequence must restore the server order, got {repaired_seqs}"
)
print("   ✅ Correct order restored — from data the client already had.")

# ── 3. Completeness: what if one never arrives at all? ──
def detect_gaps(received_seqs, last_acked=0):
    """Sequence numbers missing between last_acked and the highest one seen."""
    if not received_seqs:
        return []
    have = set(received_seqs)
    return [s for s in range(last_acked + 1, max(received_seqs) + 1) if s not in have]

assert detect_gaps(repaired_seqs) == [], (
    f"a complete run must report no gaps, got {detect_gaps(repaired_seqs)}"
)
print("   ✅ Complete run -> no gaps reported.")

dropped = repaired_seqs[len(repaired_seqs) // 2]      # pretend this push was lost
lossy = [s for s in repaired_seqs if s != dropped]
gaps = detect_gaps(lossy)
print(f"\n💥 Now drop seq={dropped} in flight. Client holds: {lossy}")
print(f"   🚨 GAPS DETECTED: {gaps}")
assert gaps == [dropped], f"expected exactly [{dropped}] missing, got {gaps}"
print(f"   -> Client re-requests seq {gaps} from the server. Nothing is lost.")

print("\n💡 Two guarantees, often confused:")
print("   • ORDERING     -> sort by sequence_number    (fixes reordering)")
print("   • COMPLETENESS -> find holes in the sequence (fixes loss)")
print("   Neither works if you sort by arrival time or by a wall clock.")
print("\n⚠️  Honest limit: a message lost at the TAIL is invisible to this check —")
print("   the client cannot miss what it never knew existed. Real clients also ask")
print("   the server 'what is your last sequence for this chat?' on reconnect.")

---

# 🔁 At-Least-Once Is Not Exactly-Once

Scroll back to the send cell. It ends with:

> *"Without this ACK, Alice's phone would keep retrying!"*

That's the correct client behaviour — and on its own it introduces a **new bug**.

```
Alice ── send_message ──▶ Server   ✅ stored as message 42
                                   │
Alice ◀───── ack ─────✂️───────────┘   ACK dies on the way back

Alice's phone: "no ACK, must have failed" ── send_message ──▶ Server
                                                              ✅ stored as message 43
Bob sees the message TWICE. 😖
```

The inbox gives us **at-least-once** delivery. Nothing in the system yet gives us
**exactly-once** — and a strict exactly-once *network* protocol is impossible
anyway (the sender can never distinguish "request lost" from "reply lost").

### The real fix: make the write idempotent

Exactly-once *effects* are achievable even over an at-least-once channel. The
recipe is always the same:

1. The **client** mints an ID for the logical send — before the first attempt.
2. Every retry reuses the **same** ID.
3. The **server** enforces uniqueness on it, so retry #2 stores nothing.

```sql
ALTER TABLE messages ADD COLUMN client_message_id TEXT;
CREATE UNIQUE INDEX ON messages (sender_id, client_message_id)
    WHERE client_message_id IS NOT NULL;
```

The key is scoped **per sender**, not globally — two users must be free to pick
the same string. Let's reproduce the duplicate, then kill it. 👇

In [ ]:
# 🔁 Duplicate-on-retry: reproduce it, then fix it

retry_content = "Did that go through? Sending again just in case 🤔"

def copies_of(content):
    return query(
        "SELECT id, sequence_number AS seq FROM messages "
        "WHERE chat_id = 1 AND content = %s ORDER BY id", (content,)
    )

# Start from a clean slate so the cell is safe to re-run.
execute("DELETE FROM inbox WHERE message_id IN "
        "(SELECT id FROM messages WHERE chat_id = 1 AND content = %s)", (retry_content,))
execute("DELETE FROM messages WHERE chat_id = 1 AND content = %s", (retry_content,))

# ── 1. Reproduce the bug against the REAL server ──
# Alice's ACK gets lost, so her phone sends the identical message again.
ws = ws_connect(WS_URL)
ws.send(json.dumps({"type": "connect", "user_id": 1}))
json.loads(ws.recv())
for attempt in (1, 2):
    ws.send(json.dumps({"type": "send_message", "chat_id": 1, "content": retry_content}))
    ack = json.loads(ws.recv())
    lost = "   ✂️ (this ACK never reaches Alice)" if attempt == 1 else ""
    print(f"   attempt {attempt}: server stored message_id={ack['message_id']}{lost}")
ws.close()

dupes = copies_of(retry_content)
dupe_ids = [d['id'] for d in dupes]
print(f"\n💥 Bob now has the same message {len(dupes)} times: ids={dupe_ids}, "
      f"seqs={[d['seq'] for d in dupes]}")
assert len(dupes) == 2, (
    f"the retry must produce a duplicate for this lesson to mean anything, "
    f"got {len(dupes)} row(s)"
)

# Roll the damage back before demonstrating the fix.
execute("DELETE FROM inbox WHERE message_id = ANY(%s)", (dupe_ids,))
execute("DELETE FROM messages WHERE id = ANY(%s)", (dupe_ids,))
# (Rewinding a counter is test-fixture housekeeping. A real server never
#  moves one backwards -- it would hand out a sequence number twice.)
execute("UPDATE chat_sequences SET last_sequence = last_sequence - %s WHERE chat_id = 1",
        (len(dupe_ids),))

# ── 2. The fix: a client-minted idempotency key + a unique index ──
execute("ALTER TABLE messages ADD COLUMN IF NOT EXISTS client_message_id TEXT")
execute("""
    CREATE UNIQUE INDEX IF NOT EXISTS idx_messages_client_msg_id
        ON messages (sender_id, client_message_id)
        WHERE client_message_id IS NOT NULL
""")
print("\n🔒 Unique index on (sender_id, client_message_id) is in place.")


def send_with_idempotency_key(chat_id, sender_id, content, client_message_id):
    """Store a message AT MOST ONCE per (sender_id, client_message_id).

    This is what the server's store_message() should do; we run the SQL here
    so the mechanism is visible. Returns (message_id, was_new).
    """
    conn = get_db()
    try:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            # Fast path: did a previous attempt already land?
            cur.execute(
                "SELECT id FROM messages WHERE sender_id = %s AND client_message_id = %s",
                (sender_id, client_message_id),
            )
            hit = cur.fetchone()
            if hit:
                return hit["id"], False        # retry -> re-ACK, store nothing

            cur.execute(
                "UPDATE chat_sequences SET last_sequence = last_sequence + 1 "
                "WHERE chat_id = %s RETURNING last_sequence", (chat_id,)
            )
            seq = cur.fetchone()["last_sequence"]
            cur.execute(
                "INSERT INTO messages (chat_id, sender_id, content, sequence_number, "
                "client_message_id) VALUES (%s, %s, %s, %s, %s) RETURNING id",
                (chat_id, sender_id, content, seq, client_message_id),
            )
            msg_id = cur.fetchone()["id"]
        conn.commit()
        return msg_id, True
    except psycopg2.errors.UniqueViolation:
        # Two retries raced past the SELECT. The INDEX -- not the SELECT -- is
        # what actually guarantees at-most-once; the SELECT is just a fast path.
        conn.rollback()
        row = query("SELECT id FROM messages WHERE sender_id = %s AND client_message_id = %s",
                    (sender_id, client_message_id))
        return row[0]["id"], False
    finally:
        conn.close()


# ── 3. Same send, three attempts, now with a stable key ──
# (This helper writes the message row only; fan-out to the inbox is the
#  server's job and is unchanged -- the dedup happens before it.)
CLIENT_MSG_ID = "notebook-retry-demo-0001"   # a UUIDv7 in a real client
results = [send_with_idempotency_key(1, 1, retry_content, CLIENT_MSG_ID)
           for _ in range(3)]
for i, (mid, was_new) in enumerate(results, 1):
    print(f"   attempt {i}: message_id={mid}  stored_new_row={was_new}")

stored = copies_of(retry_content)
print(f"\n✅ After 3 identical attempts the messages table holds it "
      f"{len(stored)} time(s): ids={[row['id'] for row in stored]}")
assert len(stored) == 1, f"idempotency key failed — {len(stored)} copies stored"
assert [w for _, w in results] == [True, False, False], (
    f"only the FIRST attempt may insert, got {[w for _, w in results]}"
)
assert len({mid for mid, _ in results}) == 1, (
    f"every retry must resolve to the same message_id, got {[m for m, _ in results]}"
)
print("   ✅ Every retry resolved to the same message_id and stored nothing new.")

# Keep this row around for the cleanup cell below.
idempotency_message_id = results[0][0]

print("\n💡 The client-minted key is the whole trick. A server-generated id can't")
print("   work: the client has to name the send BEFORE it knows if it succeeded.")

---

# 🧹 Cleanup

Let's clean up the test data we created during this notebook so the database
is back to its original state for the next notebook.

In [ ]:
# 🧹 Clean up test data created during this notebook

# NOTE: rewinding a sequence counter is test-fixture housekeeping so the next
# notebook starts from the seed state. A real server NEVER moves a counter
# backwards — that would hand out a sequence number twice.

for label, mid in (("test message", test_message_id),
                   ("idempotency demo message", idempotency_message_id)):
    if mid:
        execute("DELETE FROM inbox WHERE message_id = %s", (mid,))
        execute("DELETE FROM messages WHERE id = %s", (mid,))
        print(f"🗑️  Deleted {label} (id={mid}) and its inbox entries")

# The client_message_id column and its unique index STAY. `db/init.sql` ships
# them for fresh volumes, and the ALTER above is what back-fills an existing
# one -- dropping them here would just make the two disagree.

# Reset chat 1's sequence counter back to 3
execute("UPDATE chat_sequences SET last_sequence = 3 WHERE chat_id = 1")
print("🔄 Reset Chat 1 sequence counter to 3")

# Reset Diana's inbox back to pending (we ACK'd them during sync demo)
execute("""
    UPDATE inbox
    SET status = 'pending', delivered_at = NULL
    WHERE user_id = 4 AND message_id IN (6, 7, 8)
""")
print("🔄 Reset Diana's inbox entries back to 'pending'")

# Verify we really are back at the seed state — a leaked row would make the
# next notebook's printed observations wrong.
left = query("SELECT COUNT(*) AS n FROM messages WHERE chat_id = 1")[0]['n']
seq = query("SELECT last_sequence FROM chat_sequences WHERE chat_id = 1")[0]['last_sequence']
assert left == 3 and seq == 3, (
    f"chat 1 should be back to 3 messages / counter 3, got {left} messages / counter {seq}"
)
print("\n✅ Cleanup complete! Database is back to its original state.")

---

# 📝 Summary: Key Takeaways

Congratulations! You've explored the core of how a messaging system delivers messages. 🎉

### What we learned

| Concept | What it does | Real-world analogy |
|---------|-------------|-------------------|
| **WebSocket** | Real-time bidirectional connection | Phone call |
| **Messages table** | Permanent storage of all messages | Filing cabinet |
| **Inbox table** | Tracks what each user needs to receive | PO Box |
| **ACK (Acknowledgment)** | Confirms message was delivered | Signed receipt |
| **Redis Pub/Sub** | Instant notification to online users | Radio broadcast |
| **Sync** | Catch up on missed messages | Checking your PO Box |
| **Sequence Numbers** | Order messages *and* detect missed ones | Episode numbers on TV |
| **Idempotency key** | Makes a retried send a no-op instead of a duplicate | Order number on a reprinted invoice |

### The two-layer delivery system

```
                    Message Sent
                        │
                ┌───────┴───────┐
                │               │
                ▼               ▼
        ┌──────────────┐ ┌──────────────┐
        │  Redis       │ │   Inbox      │
        │  Pub/Sub     │ │  (Database)  │
        │              │ │              │
        │  Fast but    │ │  Slow but    │
        │  unreliable  │ │  reliable    │
        │  (at most    │ │  (at least   │
        │   once)      │ │   once)      │
        └──────────────┘ └──────────────┘
                │               │
                └───────┬───────┘
                        │
                        ▼
               Together = Reliable
               AND fast delivery! ✅
```

### ⚖️ The delivery guarantee, stated honestly

- Redis pub/sub alone is **at-most-once**.
- Inbox + ACK + resync is **at-least-once** — which means duplicates are not a
  bug, they are the *expected* behaviour of a retrying client.
- `client_message_id` + a unique index turns at-least-once *delivery* into
  exactly-once *effects*. That is the only kind of "exactly-once" that exists.
- Ordering is **per conversation**. There is no global order, and none is needed.

### 🔮 What's Next?

In **Notebook 2: Read Receipts & Presence**, we'll explore:
- How the ✓✓ (double checkmark) system works
- How the app knows when someone is "online" or "last seen at..."
- How read receipts flow back to the sender

See you there! 👋